# Misión BigMart

## Localización y variabilidad en una decisión real

**Duración:** 50 minutos  
**Herramientas:** pandas, NumPy y Plotly  
**Producto:** un dictamen de cuatro líneas respaldado por números y una gráfica.

> BigMart publicará mañana una cifra de venta típica. También debe describir qué tan variables son las ventas y decidir si eliminar registros con peso faltante es seguro.

### Reglas de trabajo

1. Trabajen en parejas: una persona es **piloto** y otra **analista**.
2. Escriban una predicción antes de ejecutar cada reto.
3. No usen **Ejecutar todo** al comenzar.
4. Cambien roles al iniciar el reto 2.
5. Cuando termine el temporizador, detengan el teclado y preparen una conclusión.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DiegoViillalba/almacenamienes-y-mineria-de-datos/blob/main/lectures/03-eda/03-03-localizacion-variabilidad-bigmart/notebook-alumnos.ipynb)

## Cómo usar este notebook

Cada reto sigue cuatro pasos:

```text
predecir → completar código → comprobar → interpretar
```

Los espacios `____` o `"________"` indican lo que deben completar. Las celdas de **control de avance** no dan la solución: confirman que el resultado es coherente.

Si abren el notebook desde una copia local del repositorio, los datos se buscan automáticamente en `data/bigmart_sales.csv`. En Colab, se utiliza la copia pública de GitHub.

## 0 · Semáforo técnico

Esta celda está completa. Ejecútenla sin modificarla.

Resultado esperado: las tres bibliotecas cargan y aparece una tabla de **8,523 filas y 12 columnas**.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly
import plotly.express as px

pd.set_option("display.max_columns", 12)


def encontrar_archivo_local():
    """Busca el CSV si el notebook se ejecuta dentro del repositorio."""
    inicio = Path.cwd().resolve()
    candidatos = [inicio / "bigmart_sales.csv"]
    candidatos.extend(
        base / "data" / "bigmart_sales.csv"
        for base in [inicio, *inicio.parents]
    )
    return next((ruta for ruta in candidatos if ruta.exists()), None)


ruta_local = encontrar_archivo_local()
URL_DATOS = (
    "https://raw.githubusercontent.com/DiegoViillalba/"
    "almacenamienes-y-mineria-de-datos/main/data/bigmart_sales.csv"
)

if ruta_local is not None:
    df = pd.read_csv(ruta_local)
    origen = str(ruta_local)
else:
    df = pd.read_csv(URL_DATOS)
    origen = "GitHub"

print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Plotly:", plotly.__version__)
print("Datos:", df.shape)
print("Origen:", origen)

df.head(3)

In [ ]:
# Control de avance 0
assert df.shape == (8523, 12), "La tabla no tiene la forma esperada. Revisa el archivo cargado."
assert "Item_Outlet_Sales" in df.columns
assert "Outlet_Type" in df.columns
print("✓ Entorno listo. No ejecutes todo: espera el primer reto.")

## Repaso de 90 segundos

- Un `DataFrame` es una tabla.
- Una `Series` es una columna.
- `df["columna"]` selecciona.
- `.agg(...)` calcula varios resúmenes.
- `.groupby(...)` separa, aplica y combina.
- En Plotly, `fig` contiene la gráfica interactiva: permite *hover*, zoom y selección.

```python
ventas = df["Item_Outlet_Sales"]
ventas.agg(["mean", "median", "std"])

df.groupby("Outlet_Type")["Item_Outlet_Sales"].median()
```

# Calentamiento · Dos rutas, el mismo promedio

Un estudiante midió su tiempo de traslado durante seis días:

- Ruta A: 38, 39, 40, 40, 41, 42 minutos.
- Ruta B: 25, 30, 38, 42, 50, 55 minutos.

**Predicción antes de calcular:** ¿cuál recomendarías a alguien que no puede llegar tarde? ¿Qué número necesitas para justificarlo?

> Nuestra predicción:  
> La evidencia que necesitamos es:

In [ ]:
ruta_a = pd.Series([38, 39, 40, 40, 41, 42], name="Ruta A")
ruta_b = pd.Series([25, 30, 38, 42, 50, 55], name="Ruta B")

# TODO: completa los métodos. Recuerda incluir paréntesis.
comparacion_rutas = pd.DataFrame({
    "media": [ruta_a.____(), ruta_b.____()],
    "desv_est": [ruta_a.____(), ruta_b.____()],
    "rango": [
        ruta_a.____() - ruta_a.____(),
        ruta_b.____() - ruta_b.____(),
    ],
}, index=["Ruta A", "Ruta B"])

comparacion_rutas.round(2)

In [ ]:
# Control de avance · calentamiento
assert np.isclose(comparacion_rutas.loc["Ruta A", "media"], 40)
assert np.isclose(comparacion_rutas.loc["Ruta B", "media"], 40)
assert comparacion_rutas.loc["Ruta A", "desv_est"] < comparacion_rutas.loc["Ruta B", "desv_est"]
print("✓ Ambas medias coinciden, pero la ruta A tiene menor variabilidad.")

# Reto 1 · ¿Cuál es la venta típica?

`Item_Outlet_Sales` contiene las ventas de un producto en un outlet. Cada fila es una observación **producto–outlet**, no una tienda completa ni una serie de tiempo.

### Predicción

Antes de calcular:

- ¿La distribución tendrá cola izquierda, será simétrica o tendrá cola derecha?
- ¿La media estará a la izquierda o a la derecha de la mediana?

> Nuestra predicción:

In [ ]:
ventas = df["Item_Outlet_Sales"].dropna()

# TODO: completa los dos métodos.
media = ventas.____()
mediana = ventas.____()

print(f"Media:   {media:,.2f} unidades monetarias")
print(f"Mediana: {mediana:,.2f} unidades monetarias")

In [ ]:
# Control de avance 1
assert np.isclose(media, 2181.288914), "Revisa cómo calculaste la media."
assert np.isclose(mediana, 1794.331), "Revisa cómo calculaste la mediana."
print("✓ Medidas de centro correctas.")

In [ ]:
# TODO: sustituye los dos ____ por la medida correspondiente.
fig_ventas = px.histogram(
    df,
    x="Item_Outlet_Sales",
    nbins=45,
    labels={"Item_Outlet_Sales": "Ventas (unidades monetarias)"},
    title="Distribución de ventas por producto–outlet",
)
fig_ventas.add_vline(
    x=____, line_color="#b45309", line_width=3,
    annotation_text=f"Media = {media:,.0f}",
)
fig_ventas.add_vline(
    x=____, line_color="#0b4a6f", line_width=3, line_dash="dash",
    annotation_text=f"Mediana = {mediana:,.0f}",
)
fig_ventas.update_layout(template="plotly_white", height=600, bargap=0.04)
fig_ventas.show(config={"displayModeBar": True, "scrollZoom": True})

### Decisión del reto 1

Completen antes de continuar:

> Para comunicar una **venta típica** usaríamos __________, cuyo valor es __________ unidades monetarias, porque la gráfica muestra __________.  
> La otra medida sería útil si la pregunta fuera __________.

# Reto 2 · Llega una venta extrema

Simularemos una observación de 100,000 unidades monetarias. No cambiaremos `df`: trabajaremos sobre una copia.

### Predicción

Ordenen de mayor a menor sensibilidad esperada:

`media` · `mediana` · `rango` · `desviación estándar` · `IQR`

> Nuestro orden esperado:

In [ ]:
con_extremo = pd.concat(
    [ventas, pd.Series([100_000.0])],
    ignore_index=True,
)


def resumen_variabilidad(serie):
    # TODO: completa cuantiles y métodos.
    q1 = serie.quantile(____)
    q3 = serie.quantile(____)
    return pd.Series({
        "n": serie.size,
        "media": serie.____(),
        "mediana": serie.____(),
        "rango": serie.____() - serie.____(),
        "desv_est": serie.____(),
        "IQR": q3 - q1,
    })


comparacion = pd.DataFrame({
    "original": resumen_variabilidad(ventas),
    "con_extremo": resumen_variabilidad(con_extremo),
})
comparacion["cambio_pct"] = (
    100 * (comparacion["con_extremo"] / comparacion["original"] - 1)
)
comparacion.round(2)

In [ ]:
# Control de avance 2
assert comparacion.loc["rango", "cambio_pct"] > 600
assert comparacion.loc["desv_est", "cambio_pct"] > 15
assert abs(comparacion.loc["mediana", "cambio_pct"]) < 0.01
assert abs(comparacion.loc["IQR", "cambio_pct"]) < 0.1
print("✓ Detectaste medidas sensibles y resistentes.")

In [ ]:
# Gráfica interactiva del impacto del extremo. Esta celda está completa.
impacto = (
    comparacion.drop(index="n")
    .rename_axis("medida")
    .reset_index()
)
impacto["factor"] = impacto["con_extremo"] / impacto["original"]
impacto["etiqueta"] = impacto["factor"].map(lambda x: f"{x:.2f}×")

fig_impacto = px.bar(
    impacto,
    x="medida",
    y="factor",
    color="cambio_pct",
    text="etiqueta",
    hover_data={"cambio_pct": ":.2f"},
    labels={
        "factor": "Valor con extremo / valor original",
        "medida": "Medida",
        "cambio_pct": "Cambio %",
    },
    color_continuous_scale="Blues",
    title="¿Cuánto cambió cada medida?",
)
fig_impacto.add_hline(y=1, line_dash="dash", line_color="#64748b")
fig_impacto.update_layout(template="plotly_white", height=600)
fig_impacto.update_traces(textposition="outside")
fig_impacto.show(config={"displayModeBar": True})

## Reto 2B · Candidatos atípicos con la regla de Tukey

Los límites son:

\[
+L_i=Q_1-1.5\,IQR, \qquad L_s=Q_3+1.5\,IQR
+\]

La regla **señala candidatos para investigar**; no demuestra que sean errores.

In [ ]:
# TODO: completa 0.25, 0.75 y la condición superior.
q1, q3 = ventas.quantile([____, ____])
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

mascara_tukey = (
    (ventas < limite_inferior)
    | (ventas > __________)
)
candidatos = ventas[mascara_tukey]

print(f"Límites: [{limite_inferior:,.2f}, {limite_superior:,.2f}]")
print(f"Candidatos: {len(candidatos)} ({len(candidatos)/len(ventas):.2%})")

In [ ]:
# Control de avance 2B
assert len(candidatos) == 186, "Revisa los cuantiles o la condición de la máscara."
print("✓ Hay 186 candidatos. Todavía no sabemos cuántos son errores.")

### Decisión del reto 2

> Las medidas que más cambiaron fueron __________.  
> Las medidas más resistentes fueron __________.  
> Antes de eliminar los 186 candidatos pediríamos __________ porque __________.

# Reto 3 · La media global oculta grupos

Pregunta: **¿cambian el centro y la dispersión de las ventas según el tipo de outlet?**

### Predicción

> Esperamos que los grupos __________ porque __________.

In [ ]:
# TODO: completa las dos columnas.
grupos = df.groupby("________")["________"]

resumen_grupos = grupos.agg(
    n="size",
    media="mean",
    mediana="median",
    desv_est="std",
)
resumen_grupos["q1"] = grupos.quantile(0.25)
resumen_grupos["q3"] = grupos.quantile(0.75)
resumen_grupos["IQR"] = resumen_grupos["q3"] - resumen_grupos["q1"]

resumen_grupos.round(2)

In [ ]:
# Control de avance 3
assert resumen_grupos["mediana"].idxmax() == "Supermarket Type3"
assert resumen_grupos["IQR"].idxmax() == "Supermarket Type3"
print("✓ Supermarket Type3 tiene la mayor mediana y el mayor IQR absoluto.")

In [ ]:
# Esta celda está completa: usen hover y zoom para explorar los grupos.
orden = resumen_grupos.sort_values("mediana").index

fig_grupos = px.box(
    df,
    x="Outlet_Type",
    y="Item_Outlet_Sales",
    color="Outlet_Type",
    points="outliers",
    category_orders={"Outlet_Type": list(orden)},
    labels={
        "Outlet_Type": "Tipo de outlet",
        "Item_Outlet_Sales": "Ventas (unidades monetarias)",
    },
    title="Centro y dispersión de ventas por tipo de outlet",
)
fig_grupos.update_layout(template="plotly_white", height=620, showlegend=False)
fig_grupos.show(config={"displayModeBar": True, "scrollZoom": True})

In [ ]:
# Una comprobación de granularidad: ¿cuántas tiendas físicas hay en cada tipo?
df.groupby("Outlet_Type")["Outlet_Identifier"].nunique().rename("numero_de_tiendas")

### Decisión del reto 3

> El grupo con mayor venta típica es __________, con una mediana de __________.  
> Su IQR indica __________.  
> No podemos afirmar __________ porque la unidad de observación es __________ y ese tipo contiene __________ tienda(s).

# Auditoría final · ¿Es seguro aplicar `dropna()`?

Debemos evaluar esta afirmación:

> “Como la media y la mediana globales cambian poco, podemos eliminar las filas con `Item_Weight` faltante.”

### Predicción

> La afirmación nos parece __________ porque todavía necesitamos revisar __________.

In [ ]:
# TODO: ¿por cuál variable categórica conviene agrupar?
pct_faltante = (
    df["Item_Weight"]
      .isna()
      .groupby(df["________"])
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

print(pct_faltante.round(2))
print("\nFilas eliminadas:", df["Item_Weight"].isna().sum())
print("Porcentaje eliminado:", f"{df['Item_Weight'].isna().mean():.2%}")

In [ ]:
# Control de avance · auditoría
assert np.isclose(pct_faltante.loc["Supermarket Type3"], 100)
assert np.isclose(pct_faltante.loc["Supermarket Type1"], 0)
assert df["Item_Weight"].isna().sum() == 1463
print("✓ El patrón faltante depende fuertemente del tipo de outlet.")

In [ ]:
# Gráfica interactiva del patrón faltante. Esta celda está completa.
tabla_faltantes = (
    pct_faltante.rename_axis("Outlet_Type")
    .reset_index(name="pct_faltante")
)

fig_faltantes = px.bar(
    tabla_faltantes,
    x="Outlet_Type",
    y="pct_faltante",
    color="pct_faltante",
    text_auto=".1f",
    labels={
        "Outlet_Type": "Tipo de outlet",
        "pct_faltante": "% faltante en Item_Weight",
    },
    color_continuous_scale="Blues",
    title="El peso faltante depende del tipo de outlet",
)
fig_faltantes.update_layout(
    template="plotly_white", height=600, coloraxis_showscale=False
)
fig_faltantes.update_traces(texttemplate="%{text}%", textposition="outside")
fig_faltantes.show(config={"displayModeBar": True})

In [ ]:
# Esta celda está completa: investiguen qué composición se pierde.
completos = df.dropna(subset=["Item_Weight"])

tiendas_originales = set(df["Outlet_Identifier"].unique())
tiendas_completas = set(completos["Outlet_Identifier"].unique())
anios_originales = set(df["Outlet_Establishment_Year"].unique())
anios_completos = set(completos["Outlet_Establishment_Year"].unique())

print("Filas originales:", len(df))
print("Filas completas:", len(completos))
print("Tiendas eliminadas:", sorted(tiendas_originales - tiendas_completas))
print("Años eliminados:", sorted(anios_originales - anios_completos))
print(
    "Tipos ausentes después de filtrar:",
    sorted(set(df["Outlet_Type"]) - set(completos["Outlet_Type"]))
)

### Veredicto de la auditoría

Seleccionen una opción y justifiquen:

- **A.** `dropna()` es seguro porque las medias globales cambian poco.
- **B.** Los faltantes son aleatorios porque aparecen en más de un tipo.
- **C.** Se puede filtrar si después se vuelve a calcular la mediana.
- **D.** La estabilidad global no basta: el filtro elimina segmentos completos.

> Elegimos __________ porque __________.

# Dictamen final

Edite y complete estas cuatro líneas. No copie código: escriba una conclusión que una persona no técnica pueda entender.

> **Decisión.** Para comunicar ____________________ usaríamos ____________________.  
> **Evidencia numérica.** Su valor es ____________________ unidades monetarias.  
> **Evidencia visual.** La gráfica muestra ____________________.  
> **Precaución.** Antes de eliminar ____________________ revisaríamos ____________________ porque ____________________.

### Comprobación entre pares

- [ ] La medida responde a una pregunta explícita.
- [ ] La cifra tiene unidad.
- [ ] La evidencia visual describe un patrón observable.
- [ ] La precaución reconoce granularidad, atípicos o faltantes.

# Actividad opcional · Del `groupby` al almacén de datos

Solo si terminan antes:

1. Encuentren la categoría modal de `Item_Type`.
2. Calculen la media de ventas de cada `Outlet_Type` y luego el promedio simple de esas cuatro medias.
3. Reconstruyan la media global usando las medias de grupo ponderadas por `n`.
4. Expliquen por qué el promedio simple de promedios no coincide con la media global.

Pista:

\[
\bar{x}=\frac{\sum_g n_g\bar{x}_g}{\sum_g n_g}
\]

In [ ]:
# TODO opcional
moda_item_type = df["________"].mode().iloc[0]

por_grupo = df.groupby("Outlet_Type")["Item_Outlet_Sales"].agg(
    n="size", media="mean"
)
promedio_de_promedios = por_grupo["media"].mean()
media_ponderada = (
    (por_grupo["n"] * por_grupo["media"]).sum()
    / por_grupo["n"].sum()
)

print("Moda de Item_Type:", moda_item_type)
print("Promedio simple de medias:", round(promedio_de_promedios, 2))
print("Media ponderada:", round(media_ponderada, 2))
print("Media global:", round(df["Item_Outlet_Sales"].mean(), 2))